# Phase 2: Uplift Modeling

Goal: estimate heterogeneous treatment effects at the user level.

In Phase 1, we validated that the experiment is randomized and balanced. In Phase 2, we move from average treatment effect to conditional treatment effect:

`uplift(x) = P(conversion | treatment=1, X=x) - P(conversion | treatment=0, X=x)`

Business translation: which users are more likely to convert because we treat them?

## 1. Setup

This first version uses scikit-learn models so the notebook runs with the current project dependencies. Later, we can swap the base model to LightGBM for stronger performance.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42

In [ ]:
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "criteo-uplift-v2.1.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "uplift_scores_sample.parquet"

assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

DATA_PATH

## 2. Load a Modeling Sample

The full dataset has nearly 14 million rows. For learning and iteration, we start with a balanced modeling sample.

Why balanced? The original experiment has many more treated users than control users. For first-pass S- and T-learners, a balanced sample makes training faster and easier to reason about. Later, we can train on larger data and use proper weighting.

In [ ]:
con = duckdb.connect(database=":memory:")

con.execute(f"""
    create or replace view experiment as
    select *
    from read_csv_auto('{DATA_PATH.as_posix()}', sample_size=100000)
""")

con.sql("select count(*) as users from experiment").df()

In [ ]:
# Start small enough to iterate quickly. Increase to 250_000+ once the notebook runs end to end.
N_PER_GROUP = 100_000

model_df = con.sql(f"""
    with control_sample as (
        select *
        from (select * from experiment where treatment = 0)
        using sample {N_PER_GROUP} rows
    ),
    treatment_sample as (
        select *
        from (select * from experiment where treatment = 1)
        using sample {N_PER_GROUP} rows
    )
    select * from control_sample
    union all
    select * from treatment_sample
""").df()

model_df.shape

In [ ]:
model_df.groupby("treatment").agg(
    users=("conversion", "size"),
    conversion_rate=("conversion", "mean"),
    visit_rate=("visit", "mean"),
    exposure_rate=("exposure", "mean"),
)

## 3. Train/Test Split

We split by rows while stratifying on treatment and conversion. This keeps treatment/outcome mix similar across train and test.

In [ ]:
feature_cols = [c for c in model_df.columns if c.startswith("f")]
target_col = "conversion"
treatment_col = "treatment"

model_df["strata"] = model_df[treatment_col].astype(str) + "_" + model_df[target_col].astype(str)

train_df, test_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=model_df["strata"],
)

train_df = train_df.drop(columns="strata")
test_df = test_df.drop(columns="strata")

train_df.shape, test_df.shape

In [ ]:
train_df.groupby(treatment_col)[target_col].agg(["size", "mean"]), test_df.groupby(treatment_col)[target_col].agg(["size", "mean"])

## 4. Helper: Model Diagnostics

AUC/average precision are not uplift metrics. They only tell us whether each response model predicts conversion reasonably. Phase 3 will evaluate uplift ranking directly with Qini/AUUC.

In [ ]:
def evaluate_response_model(y_true, y_score, label):
    return {
        "model": label,
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "mean_predicted_conversion": y_score.mean(),
        "actual_conversion_rate": y_true.mean(),
    }


def make_base_model():
    return HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.06,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )

## 5. S-Learner

S-learner = single model.

Train one conversion model using both user features and treatment assignment:

`conversion ~ features + treatment`

Then score every test user twice:

- once pretending `treatment = 1`
- once pretending `treatment = 0`

The difference is predicted uplift.

In [ ]:
s_features = feature_cols + [treatment_col]

s_model = make_base_model()
s_model.fit(train_df[s_features], train_df[target_col])

s_test_pred = s_model.predict_proba(test_df[s_features])[:, 1]
evaluate_response_model(test_df[target_col], s_test_pred, "S-learner response model")

In [ ]:
test_s_treated = test_df[feature_cols].copy()
test_s_treated[treatment_col] = 1

test_s_control = test_df[feature_cols].copy()
test_s_control[treatment_col] = 0

test_df = test_df.copy()
test_df["s_pred_treated"] = s_model.predict_proba(test_s_treated[s_features])[:, 1]
test_df["s_pred_control"] = s_model.predict_proba(test_s_control[s_features])[:, 1]
test_df["s_uplift"] = test_df["s_pred_treated"] - test_df["s_pred_control"]

test_df[["s_pred_treated", "s_pred_control", "s_uplift"]].describe()

## 6. T-Learner

T-learner = two models.

Train one model on treated users and one model on control users:

- treated model estimates `P(conversion | treatment=1, X)`
- control model estimates `P(conversion | treatment=0, X)`

Then uplift is the difference between those two predictions.

In [ ]:
treated_train = train_df[train_df[treatment_col] == 1]
control_train = train_df[train_df[treatment_col] == 0]

t_model_treated = make_base_model()
t_model_control = make_base_model()

t_model_treated.fit(treated_train[feature_cols], treated_train[target_col])
t_model_control.fit(control_train[feature_cols], control_train[target_col])

In [ ]:
treated_test = test_df[test_df[treatment_col] == 1]
control_test = test_df[test_df[treatment_col] == 0]

diagnostics = []
diagnostics.append(evaluate_response_model(
    treated_test[target_col],
    t_model_treated.predict_proba(treated_test[feature_cols])[:, 1],
    "T-learner treated outcome model",
))
diagnostics.append(evaluate_response_model(
    control_test[target_col],
    t_model_control.predict_proba(control_test[feature_cols])[:, 1],
    "T-learner control outcome model",
))

pd.DataFrame(diagnostics)

In [ ]:
test_df["t_pred_treated"] = t_model_treated.predict_proba(test_df[feature_cols])[:, 1]
test_df["t_pred_control"] = t_model_control.predict_proba(test_df[feature_cols])[:, 1]
test_df["t_uplift"] = test_df["t_pred_treated"] - test_df["t_pred_control"]

test_df[["t_pred_treated", "t_pred_control", "t_uplift"]].describe()

## 7. Compare Uplift Scores

At this stage, we inspect whether the learners produce reasonable score distributions. We are not claiming the best policy yet. That comes in Phase 3 with Qini/AUUC.

In [ ]:
uplift_summary = test_df[["s_uplift", "t_uplift"]].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
uplift_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(test_df["s_uplift"], bins=60, ax=axes[0], color="#4C78A8")
axes[0].axvline(0, color="#D62728", linestyle="--")
axes[0].set_title("S-Learner Uplift Distribution")
axes[0].set_xlabel("Predicted uplift")

sns.histplot(test_df["t_uplift"], bins=60, ax=axes[1], color="#59A14F")
axes[1].axvline(0, color="#D62728", linestyle="--")
axes[1].set_title("T-Learner Uplift Distribution")
axes[1].set_xlabel("Predicted uplift")

plt.tight_layout()

In [ ]:
test_df[["s_uplift", "t_uplift"]].corr()

## 8. Uplift Deciles

This is an early sanity check: users ranked high by predicted uplift should have stronger observed treatment-control lift than users ranked low.

This is not a perfect causal evaluation because each decile still contains observed treated/control users, but it is a useful bridge to Qini/AUUC.

In [ ]:
def decile_lift_table(df, score_col):
    scored = df.copy()
    # Decile 1 = lowest predicted uplift, decile 10 = highest predicted uplift.
    scored["uplift_decile"] = pd.qcut(
        scored[score_col].rank(method="first"),
        q=10,
        labels=False,
    ) + 1

    table = (
        scored.groupby(["uplift_decile", treatment_col])[target_col]
        .agg(users="size", conversion_rate="mean")
        .reset_index()
        .pivot(index="uplift_decile", columns=treatment_col, values=["users", "conversion_rate"])
    )
    table.columns = [f"{metric}_{'control' if treatment == 0 else 'treatment'}" for metric, treatment in table.columns]
    table = table.reset_index()
    table["observed_lift"] = table["conversion_rate_treatment"] - table["conversion_rate_control"]
    return table.sort_values("uplift_decile")


s_deciles = decile_lift_table(test_df, "s_uplift")
t_deciles = decile_lift_table(test_df, "t_uplift")

s_deciles

In [ ]:
t_deciles

In [ ]:
plt.figure(figsize=(9, 5))
sns.lineplot(data=s_deciles, x="uplift_decile", y="observed_lift", marker="o", label="S-learner")
sns.lineplot(data=t_deciles, x="uplift_decile", y="observed_lift", marker="o", label="T-learner")
plt.axhline(0, color="#D62728", linestyle="--")
plt.title("Observed Treatment-Control Lift by Predicted Uplift Decile")
plt.xlabel("Predicted uplift decile: 10 = highest uplift")
plt.ylabel("Observed conversion lift")
plt.legend()
plt.tight_layout()

## 9. Save Scored Test Set

Save the test set with uplift scores so Phase 3 can evaluate targeting policies using Qini/AUUC and cumulative gain.

In [ ]:
score_cols = [
    treatment_col,
    target_col,
    "visit",
    "exposure",
    *feature_cols,
    "s_pred_treated",
    "s_pred_control",
    "s_uplift",
    "t_pred_treated",
    "t_pred_control",
    "t_uplift",
]

test_df[score_cols].to_parquet(OUTPUT_PATH, index=False)
OUTPUT_PATH

## 10. Phase 2 Readout

Fill this in after running the notebook:

- The S-learner estimated uplift by training one conversion model with treatment as a feature.
- The T-learner estimated uplift by training separate treated and control outcome models.
- The uplift distributions show `[brief pattern]`.
- In uplift deciles, `[learner]` showed stronger separation between high- and low-uplift users.
- Phase 3 should evaluate targeting performance using Qini, AUUC, and cumulative gain curves before choosing a policy.

Important caveat: response-model AUC is not the same as uplift quality. A model can predict conversion well but still be poor at ranking incremental impact.